This homework will let you practice select query skills you learned in the class, For any variations or other features consult Snowflake Documentation anytime.

 Use Student Database and S1 schema for this homework. There are 4 tables which have data about pizzas and pizza orders from a pizza shop. Study the tables and understand the primary and foreign keys in each table. You can document them in this notebook for reference. This will help you write subsequent queries for other questions.

10 points per question.

Set up your session context

In [ ]:
-- Part A Context (login)
USE ROLE ROLE_MALLARD;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_MALLARD;
USE DB_MALLARD.HOMEWORKS;

1.	Write a query for the following 2 columns in one query. First column returns ingredients of all pizza types as Left padded strings, padded with -, to make them 100 characters long. Second column called “Cheesy” gives value with this logic. If ingredients contain cheese, then column Cheesy = Yes otherwise Cheesy = No

In [ ]:
SELECT 
    LPAD(INGREDIENTS, 100, '-') AS padded_ingredients,
    CASE 
        WHEN LOWER(INGREDIENTS) LIKE '%cheese%' THEN 'Yes'
        ELSE 'No'
    END AS Cheesy
FROM STUDENT.S1.PIZZA_TYPE;

2.	Find out most expensive large pizza price in each category

In [ ]:
SELECT 
  pt.CATEGORY,
  MAX(p.PRICE) AS max_large_price
FROM STUDENT.S1.PIZZA p
JOIN STUDENT.S1.PIZZA_TYPE pt
  ON pt.PIZZA_TYPE_ID = p.PIZZA_TYPE_ID
WHERE UPPER(p.SIZE) IN ('L', 'LARGE')   -- handles 'L' or 'Large'
GROUP BY pt.CATEGORY
ORDER BY pt.CATEGORY;

3.	How many pizza types contain garlic?

In [ ]:
SELECT 
    COUNT(*) AS garlic_pizza_types
FROM STUDENT.S1.PIZZA_TYPE
WHERE LOWER(INGREDIENTS) LIKE '%garlic%';

4.	What is average price, rounded to nearest dollar, of pizza across all sizes which contains at least one of these meats - chicken, bacon, sausage, salami

In [ ]:
SELECT 
  ROUND(AVG(p.PRICE), 0) AS avg_price_meat_pizzas
FROM STUDENT.S1.PIZZA p
JOIN STUDENT.S1.PIZZA_TYPE pt
  ON pt.PIZZA_TYPE_ID = p.PIZZA_TYPE_ID
WHERE LOWER(pt.INGREDIENTS) LIKE '%chicken%'
   OR LOWER(pt.INGREDIENTS) LIKE '%bacon%'
   OR LOWER(pt.INGREDIENTS) LIKE '%sausage%'
   OR LOWER(pt.INGREDIENTS) LIKE '%salami%';

5.	What is the most expensive order total for orders placed in Sept 2015 and how many pizzas were ordered in that order?

In [ ]:
WITH order_totals AS (
  SELECT
      po.ORDER_ID,
      SUM(p.PRICE * pod.QUANTITY)          AS order_total,
      SUM(pod.QUANTITY)                    AS total_pizzas
  FROM STUDENT.S1.PIZZA_ORDER po
  JOIN STUDENT.S1.PIZZA_ORDER_DETAIL pod
    ON pod.ORDER_ID = po.ORDER_ID
  JOIN STUDENT.S1.PIZZA p
    ON p.PIZZA_ID = pod.PIZZA_ID
  WHERE po."DATE" >= DATE '2015-09-01'
    AND po."DATE" <  DATE '2015-10-01'     -- all of Sept 2015
  GROUP BY po.ORDER_ID
)
SELECT
  ORDER_ID,
  order_total,
  total_pizzas
FROM order_totals
QUALIFY RANK() OVER (ORDER BY order_total DESC) = 1   -- return all ties at the max
ORDER BY ORDER_ID;

6.	Which month had the lowest total sales in 2015, how much was the order total in that month?

In [ ]:
WITH monthly_sales AS (
  SELECT 
      DATE_TRUNC('month', po."DATE") AS sales_month,
      SUM(p.PRICE * pod.QUANTITY)    AS total_sales
  FROM STUDENT.S1.PIZZA_ORDER po
  JOIN STUDENT.S1.PIZZA_ORDER_DETAIL pod
    ON pod.ORDER_ID = po.ORDER_ID
  JOIN STUDENT.S1.PIZZA p
    ON p.PIZZA_ID = pod.PIZZA_ID
  WHERE EXTRACT(YEAR FROM po."DATE") = 2015
  GROUP BY DATE_TRUNC('month', po."DATE")
)
SELECT 
    sales_month,
    total_sales
FROM monthly_sales
QUALIFY RANK() OVER (ORDER BY total_sales ASC) = 1   -- lowest total
ORDER BY sales_month;

7.	Which pizza category and pizza type combination is the most popular and the least popular by considering the count of that category/type pizzas ordered?

In [ ]:
WITH combo_counts AS (
  SELECT
      pt.CATEGORY,
      pt.NAME AS pizza_type,
      SUM(pod.QUANTITY) AS pizzas_ordered
  FROM STUDENT.S1.PIZZA_ORDER_DETAIL pod
  JOIN STUDENT.S1.PIZZA p
    ON p.PIZZA_ID = pod.PIZZA_ID
  JOIN STUDENT.S1.PIZZA_TYPE pt
    ON pt.PIZZA_TYPE_ID = p.PIZZA_TYPE_ID
  GROUP BY pt.CATEGORY, pt.NAME
),
ranked AS (
  SELECT
      CATEGORY,
      pizza_type,
      pizzas_ordered,
      RANK() OVER (ORDER BY pizzas_ordered DESC) AS r_desc,
      RANK() OVER (ORDER BY pizzas_ordered ASC)  AS r_asc
  FROM combo_counts
)
SELECT 'Most Popular' AS popularity, CATEGORY, pizza_type, pizzas_ordered
FROM ranked
WHERE r_desc = 1

UNION ALL

SELECT 'Least Popular' AS popularity, CATEGORY, pizza_type, pizzas_ordered
FROM ranked
WHERE r_asc = 1

ORDER BY popularity, CATEGORY, pizza_type;

8.	Label each order as lunch/dinner/late-night with the below logic and then find the number of orders for each label 
-- If order is laced from midnight to 4 PM, then it is lunch 
-- If order is placed 4 PM to 9 PM, then it is dinner
-- If order is placed between 9 PM and midnight then it is late-night.


In [ ]:
SELECT 
    CASE 
        WHEN po."TIME" < TIME '16:00:00' THEN 'Lunch'
        WHEN po."TIME" < TIME '21:00:00' THEN 'Dinner'
        ELSE 'Late-Night'
    END AS order_label,
    COUNT(*) AS number_of_orders
FROM STUDENT.S1.PIZZA_ORDER po
GROUP BY 
    CASE 
        WHEN po."TIME" < TIME '16:00:00' THEN 'Lunch'
        WHEN po."TIME" < TIME '21:00:00' THEN 'Dinner'
        ELSE 'Late-Night'
    END
ORDER BY order_label;

9.	The pizza joint owner wants one list of call categories, all pizza type names and all ingredients as a one column list to print as labels in some marketing material.  Please help her get this list in one query in one column. Hint: use union and CTE together

In [ ]:
WITH categories AS (
  SELECT DISTINCT CATEGORY AS label
  FROM STUDENT.S1.PIZZA_TYPE
  WHERE CATEGORY IS NOT NULL
),
type_names AS (
  SELECT DISTINCT NAME AS label
  FROM STUDENT.S1.PIZZA_TYPE
  WHERE NAME IS NOT NULL
),
ingredients AS (
  -- Split comma-separated INGREDIENTS into individual items
  SELECT DISTINCT TRIM(value::string) AS label
  FROM STUDENT.S1.PIZZA_TYPE,
       LATERAL FLATTEN(input => SPLIT(INGREDIENTS, ',')) f
  WHERE INGREDIENTS IS NOT NULL
)
SELECT label FROM categories
UNION
SELECT label FROM type_names
UNION
SELECT label FROM ingredients
ORDER BY label;

10.	Extract a random sampling of 10 order IDs from orders placed in Nov 2015

In [ ]:
SELECT ORDER_ID
FROM STUDENT.S1.PIZZA_ORDER
WHERE "DATE" >= DATE '2015-11-01'
  AND "DATE" <  DATE '2015-12-01'   -- ensures only November 2015
QUALIFY ROW_NUMBER() OVER (ORDER BY RANDOM()) <= 10;
